In [246]:
import requests
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import pandas as pd
import re
from tqdm import tqdm
import urllib.parse
import spacy
import scispacy
import time


In [515]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

### should be a pre-processing step before age classification
-> only for mice??

In [110]:
df_age = pd.read_csv("./model_predictions/age/age_unsloth_meta_llama_3.1_8b_doc_level_predictions.csv")

# Step 1: Filter out ranges (e.g., "8-14 weeks" or "3–4 months")
def is_range(age_str):
    return bool(re.search(r"\d+\s*[-–]\s*\d+", str(age_str)))  # includes hyphen and en-dash

df_age = df_age[
    (~df_age['prediction_encoded_label'].apply(is_range)) &
    (df_age['age_prediction'].str.contains('weeks', case=False, na=False))
].copy()

df_age.head()

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label
0,10229113,810 weeks,10229113_1,810 weeks
1,10372527,68 weeks,10372527_0,68 weeks
2,10405874,60 weeks,10405874_9,60 weeks
4,10536012,89 weeks,10536012_0,89 weeks
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult"


In [112]:
# Step 2: Split comma-separated age strings into multiple rows
df_age['prediction_encoded_label'] = df_age['prediction_encoded_label'].astype(str)
df_flat = df_age.assign(
    prediction_encoded_label_flat=df_age['prediction_encoded_label'].str.split(',')
).explode('prediction_encoded_label_flat')

df_flat.head(10)

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat
0,10229113,810 weeks,10229113_1,810 weeks,810 weeks
1,10372527,68 weeks,10372527_0,68 weeks,68 weeks
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks
4,10536012,89 weeks,10536012_0,89 weeks,89 weeks
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",46 weeks
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",adult
9,10696912,68 weeks,10696912_9,68 weeks,68 weeks
11,10814783,58 weeks,10814783_1,58 weeks,58 weeks
14,10869055,68 weeks,10869055_0,68 weeks,68 weeks
15,10900333,8 weeks,10900333_5,8 weeks,8 weeks


In [114]:
# Step 3: Clean whitespace
df_flat['prediction_encoded_label_flat'] = df_flat['prediction_encoded_label_flat'].str.strip()

# Step 4: Split into age_num and age_time
split_cols = df_flat['prediction_encoded_label_flat'].str.split(" ", n=1, expand=True)
df_flat['age_num'] = pd.to_numeric(split_cols[0], errors='coerce')
df_flat['age_time'] = split_cols[1].str.lower()  # Normalize unit
df_flat.head()

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time
0,10229113,810 weeks,10229113_1,810 weeks,810 weeks,810.0,weeks
1,10372527,68 weeks,10372527_0,68 weeks,68 weeks,68.0,weeks
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks,60.0,weeks
4,10536012,89 weeks,10536012_0,89 weeks,89 weeks,89.0,weeks
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",46 weeks,46.0,weeks


### Fix clearly invalid ranges

In [117]:
df_age_to_validate_age_too_high = df_flat[df_flat['age_num'] > 150]
df_age_to_validate_age_too_high

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time
0,10229113,810 weeks,10229113_1,810 weeks,810 weeks,810.0,weeks
16,10900347,"712 weeks, 8 weeks",10900347_22; 10900347_29; 10900347_37,"712 weeks, 8 weeks",712 weeks,712.0,weeks
30,11217864,812 weeks,11217864_27; 11217864_3,812 weeks,812 weeks,812.0,weeks
44,11859145,710 weeks,11859145_6,710 weeks,710 weeks,710.0,weeks
52,12020957,816 weeks,12020957_1,816 weeks,816 weeks,816.0,weeks
...,...,...,...,...,...,...,...
2161,38885894,810 weeks,38885894_2,810 weeks,810 weeks,810.0,weeks
2169,38996350,612 weeks,38996350_28; 38996350_32,612 weeks,612 weeks,612.0,weeks
2172,39046528,810 weeks,39046528_0; 39046528_156; 39046528_159; 390465...,810 weeks,810 weeks,810.0,weeks
2180,39142423,814 weeks,39142423_4,814 weeks,814 weeks,814.0,weeks


In [119]:
def normalize_age(age_str):
    if not age_str:
        return age_str
    
    parts = age_str.split()
    if len(parts) == 2:
        age, unit = parts
    else:
        return age_str  # Leave unchanged if not in expected format

    if "-" not in age:
        try:
            age_float = float(age)
            if age_float > 150:
                if len(age) == 3:
                    age = f"{age[0]}-{age[1:]}"
                else:
                    age = f"{age[:2]}-{age[2:]}"
        except ValueError:
            return age_str  # Not a number, leave it unchanged

    return f"{age} {unit}"

# Apply the logic
df_age_to_validate_age_too_high["prediction_encoded_label_new"] = df_age_to_validate_age_too_high["prediction_encoded_label_flat"].apply(normalize_age)
df_age_to_validate_age_too_high

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_82621/2005132760.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_age_to_validate_age_too_high["prediction_encoded_label_new"] = df_age_to_validate_age_too_high["prediction_encoded_label_flat"].apply(normalize_age)


,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,prediction_encoded_label_new
0,10229113,810 weeks,10229113_1,810 weeks,810 weeks,810.0,weeks,8-10 weeks
16,10900347,"712 weeks, 8 weeks",10900347_22; 10900347_29; 10900347_37,"712 weeks, 8 weeks",712 weeks,712.0,weeks,7-12 weeks
30,11217864,812 weeks,11217864_27; 11217864_3,812 weeks,812 weeks,812.0,weeks,8-12 weeks
44,11859145,710 weeks,11859145_6,710 weeks,710 weeks,710.0,weeks,7-10 weeks
52,12020957,816 weeks,12020957_1,816 weeks,816 weeks,816.0,weeks,8-16 weeks
...,...,...,...,...,...,...,...,...
2161,38885894,810 weeks,38885894_2,810 weeks,810 weeks,810.0,weeks,8-10 weeks
2169,38996350,612 weeks,38996350_28; 38996350_32,612 weeks,612 weeks,612.0,weeks,6-12 weeks
2172,39046528,810 weeks,39046528_0; 39046528_156; 39046528_159; 390465...,810 weeks,810 weeks,810.0,weeks,8-10 weeks
2180,39142423,814 weeks,39142423_4,814 weeks,814 weeks,814.0,weeks,8-14 weeks


### Work with PMCID

In [521]:
# Compiled regex pattern for week-related age expressions
age_keywords_pattern = r"""
\b(
    age|ages|aged|aging|old|older|young|adult|adults|senescent|
    neonatal|neonate|newborn|pup|pups|juvenile|weanling|weaning|
    postnatal|prenatal|prepubescent|fetal|fetus|fetuses|
    week[-\s]?old|weekold|                         # forms like "week-old", "weekold"
    \d+\s*(?:-|–|to)\s*\d+\s*(week|wk)s?\b|             # range: e.g., "6–8 weeks"
    \d+\s*(week|wk)s?(?!\s*old)|                   # standalone: e.g., "8 weeks" but not "8 weeks old"
    after\s+birth|post[-\s]?birth
)\b
"""

# Compile the pattern once
_age_week_regex = re.compile(age_keywords_pattern, flags=re.IGNORECASE | re.VERBOSE)

def contains_week_age_expression(text: str) -> bool:
    """
    Returns True if the text contains a week-related age expression, else False.
    """
    return bool(_age_week_regex.search(text))

In [523]:
contains_week_age_expression("The mature miR-146a (primer catalog #000468, Applied Biosystems) was normalized against the expression of U6 snRNA as an endogenous normalization control in tissue samples, while the synthetic spike-in control (cel–miR-39 mimic; Qiagen) for internal normalization in plasma samples.")

False

In [122]:
df_age_to_validate = df_flat[(df_flat['age_num'] > 40) & (df_flat['age_num'] < 150)]
df_age_to_validate

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time
1,10372527,68 weeks,10372527_0,68 weeks,68 weeks,68.0,weeks
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks,60.0,weeks
4,10536012,89 weeks,10536012_0,89 weeks,89 weeks,89.0,weeks
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",46 weeks,46.0,weeks
9,10696912,68 weeks,10696912_9,68 weeks,68 weeks,68.0,weeks
...,...,...,...,...,...,...,...
2168,38980500,68 weeks,38980500_0; 38980500_102; 38980500_105; 389805...,68 weeks,68 weeks,68.0,weeks
2173,39053427,68 weeks,39053427_4,68 weeks,68 weeks,68.0,weeks
2212,39489122,69 weeks,39489122_16; 39489122_3,69 weeks,69 weeks,69.0,weeks
2264,9916140,"56 weeks, 68 weeks",9916140_18; 9916140_34; 9916140_46; 9916140_6,"56 weeks, 68 weeks",56 weeks,56.0,weeks


In [124]:
EMAIL = "donevasimona@gmail.com"  # Your email for NCBI API access

def pmid_to_pmcid(pmid):
    url = (
        f"https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
        f"?tool=fulltext-fetcher&email={EMAIL}&ids={pmid}&format=json"
    )
    try:
        r = requests.get(url)
        data = r.json()
        records = data.get("records", [])
        if records and "pmcid" in records[0]:
            return records[0]["pmcid"]
    except Exception as e:
        print(f"  Error converting PMID {pmid}: {e}")
    return None

In [126]:
tqdm.pandas()
df_age_to_validate['PMCID'] = df_age_to_validate['PMID'].progress_apply(pmid_to_pmcid)
df_age_to_validate

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 172/172 [01:24<00:00,  2.05it/s]
/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_82621/2942715007.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_age_to_validate['PMCID'] = df_age_to_validate['PMID'].progress_apply(pmid_to_pmcid)


,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,PMCID
1,10372527,68 weeks,10372527_0,68 weeks,68 weeks,68.0,weeks,None
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks,60.0,weeks,None
4,10536012,89 weeks,10536012_0,89 weeks,89 weeks,89.0,weeks,PMC23131
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",46 weeks,46.0,weeks,None
9,10696912,68 weeks,10696912_9,68 weeks,68 weeks,68.0,weeks,None
...,...,...,...,...,...,...,...,...
2168,38980500,68 weeks,38980500_0; 38980500_102; 38980500_105; 389805...,68 weeks,68 weeks,68.0,weeks,PMC12052871
2173,39053427,68 weeks,39053427_4,68 weeks,68 weeks,68.0,weeks,None
2212,39489122,69 weeks,39489122_16; 39489122_3,69 weeks,69 weeks,69.0,weeks,None
2264,9916140,"56 weeks, 68 weeks",9916140_18; 9916140_34; 9916140_46; 9916140_6,"56 weeks, 68 weeks",56 weeks,56.0,weeks,PMC407886


In [142]:
df_age_to_validate_with_pmc = df_age_to_validate[df_age_to_validate['PMCID'].notna()].copy()
df_age_to_validate_with_pmc.shape


(26, 8)

In [144]:
df_age_to_validate_with_pmc.to_csv("./model_predictions/age/post_processing/df_age_to_validate_with_pmc.csv", index=False)

In [590]:
df_age_to_validate_with_pmc = pd.read_csv("./model_predictions/age/post_processing/df_age_to_validate_with_pmc.csv")

In [592]:
df_age_to_validate_with_pmc.head()

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,PMCID
0,10536012,89 weeks,10536012_0,89 weeks,89 weeks,89.0,weeks,PMC23131
1,12531880,"4 weeks, 68 weeks",12531880_19; 12531880_35; 12531880_46; 12531880_8,"4 weeks, 68 weeks",68 weeks,68.0,weeks,PMC151876
2,18003831,67 weeks,18003831_0,67 weeks,67 weeks,67.0,weeks,PMC6673319
3,19282478,"6 weeks, 78 weeks, 910 weeks",19282478_0; 19282478_23; 19282478_3,"6 weeks, 78 weeks, 910 weeks",78 weeks,78.0,weeks,PMC2664014
4,19850890,48 weeks,19850890_3,48 weeks,48 weeks,48.0,weeks,PMC2789629


In [594]:
def normalize_dashes(text: str) -> str:
    # Replace en dash (–), em dash (—), and minus (−) with regular hyphen (-)
    return re.sub(r"[–—−]", "-", text)

In [596]:
nlp = spacy.load("en_core_sci_sm")

In [713]:
def resolve_age_from_text(age_text_to_check: str, current_age: str, current_age_time: str) -> str:
    """
    Detects if current_age (e.g., "68") is in the text. If not, tries to detect if it was misparsed from a range like "6–8".
    Returns the matched form, or original fallback if not found.
    """
    # 1. Try common valid forms for the age
    patterns = [
        rf'\b{current_age}\b(?!\s*[%\w])',                                # e.g. 56 but NOT 56%, 56001, 56n
        rf'\b{current_age}\s*(weeks?|wks?)\b(?!\s*%)',                    # e.g. 56 weeks, not 56% weeks
        rf'\b{current_age}[-\s]?(weeks?|wks?)[-\s]?old\b(?!\s*%)',        # e.g. 56-week-old
    ]
    for pattern in patterns:
        if re.search(pattern, age_text_to_check, flags=re.IGNORECASE):
            return f"{current_age} {current_age_time}"

    # 2. Try to catch range forms like "6–8"
    if len(current_age) == 2:
        a, b = current_age[0], current_age[1]
        range_pattern = rf'\b{a}[-–]{b}(?=\s|[-–]?(weeks?|wks?)\b)'
        match = re.search(range_pattern, age_text_to_check)
        if match:
            return f"{a}-{b} {current_age_time}"

    # 3. Fallback
    return f"{current_age} {current_age_time}"


def process_row(row):
    pmcid = row['PMCID']
    current_age = str(int(row['age_num']))
    current_age_time = row['age_time']

    url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None  # Or: return row['prediction_encoded_label']

        soup = BeautifulSoup(response.text, "html.parser")

        age_sentences = []

        for section in soup.find_all("section"):
            heading = section.find(["h2", "h3", "h4"])
            heading_text = heading.get_text(strip=True) if heading else "No heading"
            if heading_text in ["Abstract", "Results", "Discussion", "Abbreviations"]:
                continue

            for p in section.find_all("p"):
                doc = nlp(p.get_text(strip=True))
                for sent in doc.sents:
                    sentence_text = sent.text.strip()
                    #label, _ = age_clf.classify(sentence_text)
                    if contains_week_age_expression(sentence_text):
                        age_sentences.append(sentence_text)

        age_text_to_check = normalize_dashes(' '.join(age_sentences))
        corrected_label = resolve_age_from_text(age_text_to_check, current_age, current_age_time)

        return corrected_label

    except Exception as e:
        print(f"Error processing {pmcid}: {e}")
        return None

In [600]:
resolve_age_from_text("C57Bl/6 and SJL/J mice (6–8 weeks old) were obtained from Jackson Laboratory (Bar Harbor, ME, USA).", "68", "weeks")

'6–8 weeks'

In [566]:
df_age_to_validate_with_pmc["prediction_encoded_label_new"] = df_age_to_validate_with_pmc.progress_apply(process_row, axis=1)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 26/26 [01:08<00:00,  2.63s/it]


In [567]:
df_age_to_validate_with_pmc

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,PMCID,prediction_encoded_label_new
0,10536012,89 weeks,10536012_0,89 weeks,89 weeks,89.0,weeks,PMC23131,8–9 weeks
1,12531880,"4 weeks, 68 weeks",12531880_19; 12531880_35; 12531880_46; 12531880_8,"4 weeks, 68 weeks",68 weeks,68.0,weeks,PMC151876,6–8 weeks
2,18003831,67 weeks,18003831_0,67 weeks,67 weeks,67.0,weeks,PMC6673319,6–7 weeks
3,19282478,"6 weeks, 78 weeks, 910 weeks",19282478_0; 19282478_23; 19282478_3,"6 weeks, 78 weeks, 910 weeks",78 weeks,78.0,weeks,PMC2664014,78 weeks
4,19850890,48 weeks,19850890_3,48 weeks,48 weeks,48.0,weeks,PMC2789629,4–8 weeks
5,19850918,56 weeks,19850918_0,56 weeks,56 weeks,56.0,weeks,PMC2794761,5–6 weeks
6,21555855,68 weeks,21555855_6; 21555855_75,68 weeks,68 weeks,68.0,weeks,PMC3104755,6–8 weeks
7,22028844,"72 weeks, 8 weeks",22028844_0; 22028844_11,"72 weeks, 8 weeks",72 weeks,72.0,weeks,PMC3197632,72 weeks
8,24973214,"68 weeks, 810 weeks",24973214_2; 24973214_3,"68 weeks, 810 weeks",68 weeks,68.0,weeks,PMC4132791,6–8 weeks
9,25183005,68 weeks,25183005_6,68 weeks,68 weeks,68.0,weeks,PMC4200254,68 weeks


### Work with Elsevier

In [568]:
def starts_with_cap_block(text, min_length=20):
    """
    Checks if the text starts with a block of ≥ `min_length` consecutive uppercase letters.

    Args:
        text (str): The input text.
        min_length (int): Minimum number of capital letters to consider a match.

    Returns:
        bool: True if it starts with ≥ `min_length` uppercase letters, False otherwise.
    """
    match = re.match(rf'^[A-Z]{{{min_length},}}', text.strip())
    return bool(match)

def is_reference_line(text):
    """
    Determines whether a line resembles a citation or reference entry.

    Args:
        text (str): A line or text chunk.

    Returns:
        bool: True if it's likely a reference, False otherwise.
    """
    text = text.strip()

    # Common: Number + year + page numbers (e.g., "825 1999 189 193")
    if re.search(r'\b\d{3,4}\s+\d{4}\s+\d{2,4}\s+\d{2,4}\b', text):
        return True

    # Contains "et al." + year
    if re.search(r'\bet al\.,?\s+\d{4}', text):
        return True

    # 2 or more people with initials, e.g., "J.L. Ferrara P. Morell"
    if len(re.findall(r'\b[A-Z]\.[A-Z]?\.?\s+[A-Z][a-z]+', text)) >= 2:
        return True

    # Journal name or abbreviation at the end (e.g., "Ann.")
    if re.search(r'\b[A-Z][a-z]{1,}\.$', text):
        return True

    # Starts with a number and has a comma-year author pattern
    if re.match(r'\d{3,4}', text) and re.search(r'[A-Z][a-z]+,\s*\d{4}', text):
        return True

    return False

def is_numeric_metadata_line(text):
    """
    Detects lines that are likely metadata or serial data (mostly numbers).

    Args:
        text (str): Input line or chunk.

    Returns:
        bool: True if it's mostly numeric/ID-like, False if not.
    """
    text = text.strip()
    tokens = text.split()

    # Case: starts with "serial" or "JL", followed by mostly digits
    if tokens and tokens[0].lower() in {"serial", "jl"}:
        return True

    # If more than 80% of tokens are numbers
    numeric_tokens = sum(t.isdigit() for t in tokens)
    if len(tokens) > 0 and numeric_tokens / len(tokens) > 0.8:
        return True

    # If there are 5+ number tokens and no verbs or sentence structure
    if numeric_tokens >= 5:
        return True

    return False

In [602]:
df_age_to_validate_no_pmc = df_age_to_validate[~df_age_to_validate['PMCID'].notna()].copy()
df_age_to_validate_no_pmc.head()

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,PMCID
1,10372527,68 weeks,10372527_0,68 weeks,68 weeks,68.0,weeks,None
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks,60.0,weeks,None
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",46 weeks,46.0,weeks,None
9,10696912,68 weeks,10696912_9,68 weeks,68 weeks,68.0,weeks,None
11,10814783,58 weeks,10814783_1,58 weeks,58 weeks,58.0,weeks,None


In [604]:
def pmid_to_doi(pmid):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        "db": "pubmed",
        "id": pmid,
        "retmode": "xml"
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error fetching PMID {pmid}")
        return None

    root = ET.fromstring(response.content)
    for article_id in root.findall(".//ArticleId"):
        if article_id.attrib.get("IdType") == "doi":
            return article_id.text
    return None

In [606]:
def doi_to_html_url(doi):
    return f"https://doi.org/{doi}"  # Redirects to publisher page

In [608]:
doi_value = pmid_to_doi("10372527")

In [609]:
def get_elsevier_full_text(pii, api_key):
    """
    Fetches the full article text from Elsevier API using the given PII and API key.

    Args:
        pii (str): Publisher Item Identifier for the article.
        api_key (str): Your Elsevier API key.
        current_age (str): Fallback string if retrieval fails.
        current_age_time (str): Fallback string if retrieval fails.

    Returns:
        str: Full article text if successful, otherwise fallback message.
    """
    url = f"https://api.elsevier.com/content/article/pii/{pii}"
    headers = {
        "X-ELS-APIKey": api_key,
        "Accept": "application/json"
    }

    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        return None

    try:
        data = res.json()
        return data['full-text-retrieval-response']['originalText']
    except Exception:
        return None

In [612]:
def extract_springer_article_text(html_text):
    """
    Extracts the main article content from Springer HTML page.

    Args:
        html_text (str): The raw HTML content of the Springer article page.

    Returns:
        str or None: Cleaned article text if found, otherwise None.
    """
    soup = BeautifulSoup(html_text, 'html.parser')
    article_div = soup.find('div', {'class': 'c-article-body'})
    
    if article_div:
        return article_div.get_text(separator='\n', strip=True)
    
    return None

In [614]:
def get_content_with_selenium(url: str, wait_time: int = 5) -> str:
    options = Options()
    options.add_argument("--headless")  # Run in headless mode
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0")

    driver = webdriver.Chrome(options=options)
    
    try:
        driver.get(url)
        time.sleep(wait_time)  # Let JS challenge resolve
        page_source = driver.page_source
    finally:
        driver.quit()

    soup = BeautifulSoup(page_source, 'html.parser')
    article_div = soup.find('div', class_='article-body')  # or whatever the body class is
    
    if article_div:
        text = article_div.get_text(separator="\n", strip=True)
        return text
    else:
        return None
    

In [743]:
def get_normalized_age_from_elsevier(pmid: str, current_age: str, current_age_time: str, api_key: str) -> str:
    """
    Retrieves article text from Elsevier API using PMID, finds age-related sentences,
    and returns a normalized age label like "8-10 week".
    """
   
    # Step 1: Get DOI
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; DOI-fetcher/1.0; donevasimona@gmail.com)"
    }
    time.sleep(1.5)  # Delay for NCBI API
    doi = pmid_to_doi(pmid)
    if not doi:
        print(f"no doi found for {pmid}")
        return f"{current_age} {current_age_time}"

    # Step 2: Get ScienceDirect redirect URL
    response = requests.get(doi_to_html_url(doi), headers=headers, allow_redirects=True)
    sciencedirect_url = response.url

    # Step 3: Extract PII if possible
    match = re.search(r'/pii/([A-Z0-9]+)', sciencedirect_url)
    
    if not match:
        # Step 4: Get full text directly
        print(f"processing {pmid} with bsoup")
        original_text = extract_springer_article_text(response.text)

        # try with selenium 
        if not original_text:
            print(f"processing {pmid} with selenium {response.url}")
            original_text = get_content_with_selenium(response.url)    
    else:
        # Step 4: Get full text from Elsevier API
        pii = match.group(1)
        print(f"processing  {pmid} with Elsevier {pii}")
        original_text = get_elsevier_full_text(pii, api_key)
        
    if not original_text:
        print(f"no original text found for: {pmid}")
        return f"{current_age} {current_age_time}"

    # Step 5: Extract and classify age-related sentences
    #print(original_text)
    doc = nlp(original_text)
    age_sentences = []

    for sent in doc.sents:
        sentence_text = sent.text.strip()
        if starts_with_cap_block(sentence_text):
            continue
        if is_numeric_metadata_line(sentence_text):
            continue
        #label, _ = age_clf.classify(sentence_text)
        if contains_week_age_expression(sentence_text):
            age_sentences.append(sentence_text)

    # Step 6: Normalize and return
    age_text_to_check = normalize_dashes(' '.join(age_sentences))
    #print(age_text_to_check)
    return resolve_age_from_text(age_text_to_check, current_age, current_age_time)

In [745]:
def load_elsevier_api_key(filepath="api_keys.txt"):
    with open(filepath, "r") as f:
        for line in f:
            if line.startswith("elsevier_api_key_uoz="):
                return line.strip().split("=")[1].strip('"')
    raise ValueError("API key not found in file.")

API_KEY = load_elsevier_api_key("../07_full_text_retrieval/api_keys.txt")


In [747]:
df_age_to_validate_no_pmc["prediction_encoded_label_new"] = df_age_to_validate_no_pmc.progress_apply(
    lambda row: get_normalized_age_from_elsevier(
        pmid=row["PMID"],
        current_age=str(int(row["age_num"])),
        current_age_time=row["age_time"],
        api_key=API_KEY
    ),
    axis=1
)

  0%|                                                                                                                                             | 0/146 [00:00<?, ?it/s]

processing  10372527 with Elsevier S0730725X98002161


  1%|█▊                                                                                                                                   | 2/146 [00:04<04:48,  2.01s/it]

processing  10405874 with Elsevier S0192056199000193


  2%|██▋                                                                                                                                  | 3/146 [00:08<07:39,  3.21s/it]

...

 99%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ | 145/146 [14:46<00:05,  5.75s/it]

processing  39053427 with Elsevier S0753332224010722


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 146/146 [14:52<00:00,  5.79s/it]

processing  39489122 with Elsevier S0753332224015397


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 146/146 [14:57<00:00,  6.15s/it]


In [749]:
df_age_to_validate_no_pmc

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,PMCID,prediction_encoded_label_new
1,10372527,68 weeks,10372527_0,68 weeks,68 weeks,68.0,weeks,None,6-8 weeks
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks,60.0,weeks,None,60 weeks
7,10613826,"46 weeks, adult",10613826_0; 10613826_21,"46 weeks, adult",46 weeks,46.0,weeks,None,4-6 weeks
9,10696912,68 weeks,10696912_9,68 weeks,68 weeks,68.0,weeks,None,6-8 weeks
11,10814783,58 weeks,10814783_1,58 weeks,58 weeks,58.0,weeks,None,5-8 weeks
...,...,...,...,...,...,...,...,...,...
2107,38155065,68 weeks,38155065_42,68 weeks,68 weeks,68.0,weeks,None,6-8 weeks
2134,38411708,"68 weeks, 8 weeks",38411708_144; 38411708_149; 38411708_154; 3841...,"68 weeks, 8 weeks",68 weeks,68.0,weeks,None,6-8 weeks
2154,38761250,68 weeks,38761250_174; 38761250_182; 38761250_190; 3876...,68 weeks,68 weeks,68.0,weeks,None,6-8 weeks
2173,39053427,68 weeks,39053427_4,68 weeks,68 weeks,68.0,weeks,None,6-8 weeks


In [751]:
len(df_age_to_validate_no_pmc[df_age_to_validate_no_pmc['prediction_encoded_label_flat']==df_age_to_validate_no_pmc['prediction_encoded_label_new']])

17

In [753]:
df_age_to_validate_no_pmc[df_age_to_validate_no_pmc['prediction_encoded_label_flat']==df_age_to_validate_no_pmc['prediction_encoded_label_new']]

,PMID,age_prediction,supporting_sentence_ids,prediction_encoded_label,prediction_encoded_label_flat,age_num,age_time,PMCID,prediction_encoded_label_new
2,10405874,60 weeks,10405874_9,60 weeks,60 weeks,60.0,weeks,None,60 weeks
136,15845917,68 weeks,15845917_15; 15845917_52,68 weeks,68 weeks,68.0,weeks,None,68 weeks
263,18563891,68 weeks,18563891_8,68 weeks,68 weeks,68.0,weeks,None,68 weeks
361,20233106,68 weeks,20233106_26,68 weeks,68 weeks,68.0,weeks,None,68 weeks
416,21233144,79 weeks,21233144_31,79 weeks,79 weeks,79.0,weeks,None,79 weeks
442,21459808,68 weeks,21459808_0,68 weeks,68 weeks,68.0,weeks,None,68 weeks
538,22642808,68 weeks,22642808_0,68 weeks,68 weeks,68.0,weeks,None,68 weeks
596,23232603,68 weeks,23232603_0,68 weeks,68 weeks,68.0,weeks,None,68 weeks
640,23660634,68 weeks,23660634_1; 23660634_18,68 weeks,68 weeks,68.0,weeks,None,68 weeks
835,25576296,68 weeks,25576296_3,68 weeks,68 weeks,68.0,weeks,None,68 weeks


In [645]:
def test_text_processing_age(original_text):
    doc = nlp(original_text)
    age_sentences = []
    
    for sent in doc.sents:
        sentence_text = sent.text.strip()
        if starts_with_cap_block(sentence_text):
            continue
        if is_numeric_metadata_line(sentence_text):
            continue
        if contains_week_age_expression(sentence_text):
            age_sentences.append(sentence_text)
            print(sentence_text)

    return " ".join(age_sentences)

In [723]:
pmid = "19796825"
doi = pmid_to_doi(pmid)
if not doi:
    print(f"no doi found for {pmid}")

# Step 2: Get ScienceDirect redirect URL
response = requests.get(doi_to_html_url(doi), headers=headers, allow_redirects=True)
sciencedirect_url = response.url

# Step 3: Extract PII
match = re.search(r'/pii/([A-Z0-9]+)', sciencedirect_url)
if not match:
    print(f"no sciencedirect_url found for {pmid}")

In [725]:
response.url

'https://linkinghub.elsevier.com/retrieve/pii/S0165572809003397'

In [721]:
url = response.url
html = get_html_with_selenium(url)

soup = BeautifulSoup(html, 'html.parser')
article_div = soup.find('div', class_='article-body')  # or whatever the body class is

if article_div:
    original_text = article_div.get_text(separator="\n", strip=True)
    test_text_processing_age(text)
else:
    print("Article body not found.")

Article body not found.


In [727]:
pii = match.group(1)
print("processing ", pii)
original_text = get_elsevier_full_text(pii, API_KEY)

processing  S0165572809003397


In [707]:
#original_text

In [729]:
age_txt = test_text_processing_age(original_text)

Keywords EAE Hsp70 Inflammation NF-κB Oral Triptolide 1 Introduction Approximately 2 million people worldwide, and about 350,000 people in the United States alone have multiple sclerosis (MS), the most common debilitating illness among young adults (Zivadinov et al., 2003).
In brief, 6–7week old SJL/J female mice (Jackson Laboratory, Bar Harbor, ME) were immunized by subcutaneous injection in the right flank with an emulsion containing 150μg proteolipid protein PLP139–151 peptide in complete Freund's adjuvant (CFA) containing 200μg M. tuberculosis.
2.4 Lymph node cells stimulation Lymph node cells (LNCs) were obtained from naïve SJL mice (7weeks old).
2.7 Cell culture The murine macrophage cell line RAW264.7 (American Type Cell Collection, Rockville, MD) was maintained in DMEM (Invitrogen) supplemented with 10% fetal bovine serum and antibiotics, at 37°C in 5% CO2 incubator.


In [731]:
resolve_age_from_text(age_txt, "67", "weeks")

'6-7 weeks'

In [455]:
soup = BeautifulSoup(response.text, 'html.parser')

# Find the main article body (Springer articles often use this class)
article_div = soup.find('div', {'class': 'c-article-body'})

# Extract and clean the text
if article_div:
    article_text = article_div.get_text(separator='\n', strip=True)
    doc = nlp(article_text)
    age_sentences = []

    for sent in doc.sents:
        sentence_text = sent.text.strip()
        #label, _ = age_clf.classify(sentence_text)
        if contains_week_age_expression(sentence_text):
            age_sentences.append(sentence_text)
else:
    print("Article body not found.")

Article body not found.


In [273]:
age_sentences

['MS affects approximately 2.8 million individuals each year and is the leading cause of non-traumatic disability in young and middle-aged people [\n2\n].',
 'EAE Induction and 4-OI Intervention\nEAE was induced in female C57BL/6 mice aged 6–8\xa0weeks.',
 'BV2 Microglia Inflammation Model Induction and 4-OI Intervention\nBV2 microglia were cultivated in Dulbecco’s Modified Eagle Medium supplemented with 10% fetal bovine serum at 37 °C with 5% CO\n2\n.',
 'Article\nPubMed\nGoogle Scholar\nKim, Byung-Chul., Woo-Kwang. Jeon, Hye-Young.',
 'Hahn, Young-Myeong.']